# FlashRank Reranker

FlashRank는 **초경량이면서 초고속**으로 작동하는 오픈소스 Reranker 라이브러리입니다. 로컬에서 빠르게 재정렬이 필요한 경우에 이상적인 선택입니다.

## 학습 목표

- FlashRank의 특징과 장점 이해
- 경량 Reranker 모델 활용법 습득
- LangChain FlashrankRerank 통합 구현
- 속도 중심 Reranker 선택 전략 학습

## FlashRank의 특징

- ✅ **초고속**: 가장 빠른 재정렬 속도
- ✅ **초경량**: 최소 메모리 사용
- ✅ **무료**: 완전 오픈소스 (로컬 실행)
- ✅ **간편 설치**: 단순한 의존성
- ✅ **Production Ready**: 안정적인 성능


## 1. 환경 설정

### 1.1 FlashRank 설치

```bash
pip install flashrank
```

### 1.2 FlashRank 모델

- `ms-marco-MultiBERT-L-12`: 다국어 지원 (권장)
- `ms-marco-MiniLM-L-12-v2`: 영어 특화, 높은 정확도
- `ms-marco-TinyBERT-L-2-v2`: 최고 속도, 경량

**특징**:
- API 키 불필요 (완전 로컬)
- 자동 모델 다운로드
- CPU에서도 빠른 성능


In [ ]:
# 필요한 라이브러리 import
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from flashrank import Ranker

# 모델 재빌드 (flashrank의 Ranker 등록 필요)
FlashrankRerank.model_rebuild()

# 환경 변수 로드
load_dotenv()

## 2. 문서 출력 도우미 함수


In [ ]:
def pretty_print_docs(docs, show_scores=True):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        
        # FlashRank는 relevance_score를 metadata에 추가
        if show_scores and 'relevance_score' in doc.metadata:
            score = doc.metadata['relevance_score']
            print(f"관련성 점수: {score:.4f}")
        
        # 문서 ID 표시 (있는 경우)
        if 'id' in doc.metadata:
            print(f"문서 ID: {doc.metadata['id']}")
        
        print(f"내용: {doc.page_content}")
        print(f"{'─' * 100}\n")


## 3. 기본 검색 시스템 구축


In [ ]:
# 1단계: 문서 로드
documents = TextLoader("./data/appendix-keywords.txt").load()

print(f"✅ 문서 로드 완료")
print(f"   - 문서 수: {len(documents)}")


In [ ]:
# 2단계: 텍스트 분할 및 ID 추가
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

texts = text_splitter.split_documents(documents)

# 각 청크에 고유 ID 추가 (FlashRank는 ID를 사용)
for idx, text in enumerate(texts):
    text.metadata["id"] = idx

print(f"✅ 문서 분할 완료")
print(f"   - 청크 수: {len(texts)}")
print(f"   - 각 청크에 ID 추가 완료")


In [ ]:
# 3단계: 벡터스토어 생성
vectorstore = FAISS.from_documents(texts, OpenAIEmbeddings())
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}  # 초기 검색: 10개 문서
)

print("✅ 벡터스토어 생성 완료")
print(f"   - 임베딩: OpenAI")
print(f"   - 초기 검색 수: 10개")


In [ ]:
# 기본 검색 테스트
query = "프로그래밍에서 코드를 분석하는 도구는?"

print(f"\n🔍 검색 쿼리: {query}\n")

# Reranker 적용 전 기본 검색
docs_before = base_retriever.invoke(query)

print("📊 [Reranker 적용 전] 벡터 유사도 기반 검색 결과:")
print(f"\n상위 3개 문서 ID: {[doc.metadata.get('id', 'N/A') for doc in docs_before[:3]]}")
pretty_print_docs(docs_before[:3], show_scores=False)


## 4. FlashRank Reranker 적용

FlashRank를 사용하여 빠르게 재정렬합니다.


In [ ]:
# ---------------------------------------------------
# FlashRank Reranker 설정 — 초경량 초고속 재정렬
# ---------------------------------------------------
# ============================================================
# TODO: FlashrankRerank와 ContextualCompressionRetriever로 재정렬 파이프라인을 만드세요
# 힌트: FlashrankRerank(model="ms-marco-MultiBERT-L-12", top_n=3)
# 예상 결과: "FlashRank Reranker 설정 완료" 출력
# ============================================================


In [ ]:
# ---------------------------------------------------
# FlashRank Reranker 적용 후 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: compression_retriever로 쿼리를 검색하고 실행 시간과 결과를 확인하세요
# 힌트: time.time()으로 실행 시간 측정, pretty_print_docs(docs, show_scores=True)
# 예상 결과: 실행 시간과 3개 문서의 관련성 점수 출력
# ============================================================


## 5. 결과 비교 및 분석


In [ ]:
# ---------------------------------------------------
# FlashRank Reranker 적용 전후 결과 비교 분석
# ---------------------------------------------------
# ============================================================
# TODO: docs_before와 docs_after를 비교하여 FlashRank 효과를 분석하세요
# 힌트: 적용 전 상위 3개와 적용 후 상위 3개의 문서 ID, 내용, 관련성 점수를 비교
# 예상 결과: 적용 전/후 비교와 FlashRank 강점 출력
# ============================================================


## 6. 다양한 쿼리 테스트


In [ ]:
# ---------------------------------------------------
# 다양한 쿼리로 FlashRank 성능 검증
# ---------------------------------------------------
# ============================================================
# TODO: 여러 쿼리를 순회하며 FlashRank로 검색하고 실행 시간과 관련성 점수를 확인하세요
# 힌트: for 루프로 test_queries 순회, time.time()으로 속도 측정
# 예상 결과: 각 쿼리별 실행 시간과 상위 3개 문서 출력
# ============================================================


## 7. 핵심 정리

### 전체 Reranker 비교

| 특징 | FlashRank | Cross-Encoder | Jina | Cohere |
|------|:---:|:---:|:---:|:---:|
| 속도 | 최고 | 보통 | 빠름 | 빠름 |
| 정확도 | 보통 | 높음 | 높음 | 최고 |
| 비용 | 무료 | 무료 | $ | $$ |
| 토큰 길이 | 512 | 512 | 8K | 4K |
| 실행 환경 | 로컬 (CPU) | 로컬 (GPU 권장) | 클라우드 | 클라우드 |
| 추천 용도 | 빠른 프로토타입 | 개발/테스트 | 긴 문서 | 프로덕션 |

### 기본 사용법

```python
from langchain.retrievers.document_compressors import FlashrankRerank

compressor = FlashrankRerank(
    model="ms-marco-MultiBERT-L-12",
    top_n=3,
)
```

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | FlashRank 모델명 | `ms-marco-MultiBERT-L-12` |
| `top_n` | 최종 반환 문서 수 | 3 |